In [1]:
import json
import pandas as pd
import numpy as np

In [2]:
# Load raw flood indicator fields from VCP
vcp_path = "data_sources/hazards/VCP_Tracts.geojson"
with open(vcp_path) as f:
    vcp_data = json.load(f)

records = []
for feat in vcp_data["features"]:
    props = feat["properties"]
    records.append({
        "GEOID": str(props["GEOID"]),
        "flood_bam_100_pct": props.get("Flood_BAM_100_pct"),
        "flood_bam_500_pct": props.get("Flood_BAM_500_pct"),
        "flood_verywet_pre_pct": props.get("Flood_verywet_pre_pct"),
        "flood_verywet_fut_pct": props.get("Flood_verywet_fut_pct"),
    })

df = pd.DataFrame(records)
print(f"{len(df)} tracts loaded")
df.describe()

9106 tracts loaded


,flood_bam_100_pct,flood_bam_500_pct,flood_verywet_pre_pct,flood_verywet_fut_pct
count,9098.000000,9098.000000,9098.000000,9098.000000
mean,4.554663,15.710876,23.635374,25.657205
std,13.709503,29.817501,2.431114,2.756402
min,0.000000,0.000000,12.248260,12.538252
25%,0.000000,0.000000,22.100680,23.724760
50%,0.000000,0.987248,23.487914,25.380799
75%,2.608696,12.273199,25.131846,27.532180
max,100.000000,100.000000,33.892827,34.741693


In [3]:
# BAM values are percentages (0–100) representing the share of LandScan squares
# within each tract that fall inside the floodplain. VCP does not re-normalize BAM
# before summing because it is already on a 0–1 scale. Divide by 100 to restore that.
df["bam_100"] = df["flood_bam_100_pct"] / 100
df["bam_500"] = df["flood_bam_500_pct"] / 100

In [4]:
# Normalize very-wet-days percentages using min-max across BOTH periods combined.
# VCP: "When there are two periods involved, now and projected, the scales for the
# two values are combined before normalization. The maximum is the largest value in
# either period and the minimum is the lowest value in either period."
verywet_all = pd.concat([df["flood_verywet_pre_pct"], df["flood_verywet_fut_pct"]], ignore_index=True)
vw_min = verywet_all.min()
vw_max = verywet_all.max()
print(f"Very wet days range across both periods: {vw_min:.4f} – {vw_max:.4f}")

df["verywet_pre_norm"] = (df["flood_verywet_pre_pct"] - vw_min) / (vw_max - vw_min)
df["verywet_fut_norm"] = (df["flood_verywet_fut_pct"] - vw_min) / (vw_max - vw_min)

Very wet days range across both periods: 12.2483 – 34.7417


In [5]:
# Compute raw composite flood scores following VCP methodology:
#   score = BAM (0–1, not re-normalized) + 0.5 * verywet_norm (0–1)
# VCP: "because flood risk is primarily driven by local topography and regional
# drainage, we feel that the BAM data should have an overwhelming influence on our
# flood risk score. To ensure that it does, we reduce the weight of this variable
# [very wet days] by half."
df["flood_hazard_raw"]     = df["bam_100"] + 0.5 * df["verywet_pre_norm"]
df["flood_hazard_fut_raw"] = df["bam_500"] + 0.5 * df["verywet_fut_norm"]

In [6]:
# Normalize each index to 0–100 using min-max (consistent with heat_hazard_idx_norm)
def normalize_0_100(series):
    s_min = series.min()
    s_max = series.max()
    return ((series - s_min) / (s_max - s_min)) * 100

df["flood_hazard_idx_norm"]     = normalize_0_100(df["flood_hazard_raw"])
df["flood_hazard_fut_idx_norm"] = normalize_0_100(df["flood_hazard_fut_raw"])

df[["flood_hazard_idx_norm", "flood_hazard_fut_idx_norm"]].describe()

,flood_hazard_idx_norm,flood_hazard_fut_idx_norm
count,9098.000000,9098.000000
mean,22.767837,30.912644
std,10.934874,20.617141
min,0.000000,0.000000
25%,17.625607,19.115814
50%,20.418565,23.071316
75%,24.191176,30.278736
max,100.000000,100.000000


In [7]:
output = df[[
    "GEOID",
    "flood_bam_100_pct",
    "flood_bam_500_pct",
    "flood_verywet_pre_pct",
    "flood_verywet_fut_pct",
    "flood_hazard_idx_norm",
    "flood_hazard_fut_idx_norm",
]]
output.to_csv("data/hazards/flood_hazard.csv", index=False)
print(f"Wrote {len(output)} rows → data/hazards/flood_hazard.csv")
output.head()

Wrote 9106 rows → data/hazards/flood_hazard.csv


,GEOID,flood_bam_100_pct,flood_bam_500_pct,flood_verywet_pre_pct,flood_verywet_fut_pct,flood_hazard_idx_norm,flood_hazard_fut_idx_norm
0,06037137504,0.000000,0.000000,28.952255,28.859300,28.305344,24.992923
1,06037138000,0.000000,0.000000,29.090136,29.742482,28.538987,26.345367
2,06037139200,0.645161,0.645161,26.673860,28.785350,24.936362,25.324132
3,06087120901,5.833333,6.041667,28.531081,29.176544,32.038485,29.640819
4,06087120902,4.400978,5.134474,28.286299,29.018764,30.531792,28.774243
